In [ ]:
# FraudGuard — Hard-Fraud Feature Engineering

## Day 6

### Objective

Improve detection of difficult fraud cases identified during Day 5 error analysis.

### Day 4 XGBoost Benchmark

- ROC-AUC: 0.9049
- PR-AUC: 0.5391
- Best tested F1: 0.5200
- Threshold: 0.30
- Precision: 62.82%
- Recall: 44.36%

### Day 5 Finding

Most false negatives are not simple threshold errors. A large proportion receive very low fraud probabilities, indicating that the model needs better feature representation and interaction signals.

### Day 6 Strategy

Create targeted, interpretable features rather than blindly increasing model complexity.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.sparse import load_npz
import joblib

PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)

Project root: /Users/ayushkumar/Desktop/Fraudguard


In [2]:
X_train_final = load_npz(
    PROCESSED_DIR / "X_train.npz"
)

X_validation_final = load_npz(
    PROCESSED_DIR / "X_validation.npz"
)

y_train = pd.read_csv(
    PROCESSED_DIR / "y_train.csv"
).squeeze()

y_validation = pd.read_csv(
    PROCESSED_DIR / "y_validation.csv"
).squeeze()

print("Training:", X_train_final.shape)
print("Validation:", X_validation_final.shape)

Training: (484847, 891)
Validation: (105693, 891)


In [3]:
train_transaction = pd.read_csv(
    DATA_RAW / "train_transaction.csv"
)

train_identity = pd.read_csv(
    DATA_RAW / "train_identity.csv"
)

full_train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Full dataset:", full_train.shape)

Full dataset: (590540, 434)


In [4]:
SECONDS_PER_DAY = 24 * 60 * 60

full_train["transaction_day"] = (
    full_train["TransactionDT"] // SECONDS_PER_DAY
)

TRAIN_END_DAY = 145

train_original = full_train[
    full_train["transaction_day"] <= TRAIN_END_DAY
].copy()

validation_original = full_train[
    full_train["transaction_day"] > TRAIN_END_DAY
].copy()

print("Training rows:", len(train_original))
print("Validation rows:", len(validation_original))

Training rows: 484847
Validation rows: 105693


In [5]:
def create_v258_features(df):

    df = df.copy()

    df["V258_high_risk"] = (
        df["V258"] > 2
    ).astype("int8")

    df["V258_medium_risk"] = (
        (df["V258"] > 1) &
        (df["V258"] <= 2)
    ).astype("int8")

    df["V258_missing"] = (
        df["V258"].isna()
    ).astype("int8")

    return df

In [6]:
train_engineered = create_v258_features(
    train_original
)

validation_engineered = create_v258_features(
    validation_original
)

print(
    train_engineered[
        [
            "V258_high_risk",
            "V258_medium_risk",
            "V258_missing"
        ]
    ].sum()
)

V258_high_risk        4941
V258_medium_risk      9329
V258_missing        373094
dtype: int64


In [7]:
def create_v294_features(df):

    df = df.copy()

    df["V294_high"] = (
        df["V294"] > 1
    ).astype("int8")

    df["V294_missing"] = (
        df["V294"].isna()
    ).astype("int8")

    return df

In [8]:
train_engineered = create_v294_features(
    train_engineered
)

validation_engineered = create_v294_features(
    validation_engineered
)

In [9]:
new_features = [
    "V258_high_risk",
    "V258_medium_risk",
    "V258_missing",
    "V294_high",
    "V294_missing"
]

display(
    train_engineered[new_features].describe()
)

,V258_high_risk,V258_medium_risk,V258_missing,V294_high,V294_missing
count,484847.000000,484847.000000,484847.000000,484847.000000,484847.000000
mean,0.010191,0.019241,0.769509,0.067409,0.000025
std,0.100434,0.137372,0.421148,0.250729,0.004975
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,0.000000,0.000000
50%,0.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.000000,1.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000


In [10]:
print("New features:")

for feature in new_features:
    print(
        feature,
        "→",
        train_engineered[feature].sum()
    )

New features:
V258_high_risk → 4941
V258_medium_risk → 9329
V258_missing → 373094
V294_high → 32683
V294_missing → 12


In [11]:
# Remove target and ID
X_train_day6 = train_engineered.drop(
    columns=["isFraud", "TransactionID"]
)

X_validation_day6 = validation_engineered.drop(
    columns=["isFraud", "TransactionID"]
)

print("Day 6 raw training shape:", X_train_day6.shape)
print("Day 6 raw validation shape:", X_validation_day6.shape)

Day 6 raw training shape: (484847, 438)
Day 6 raw validation shape: (105693, 438)


In [12]:
numerical_features_day6 = X_train_day6.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features_day6 = X_train_day6.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_features_day6))
print("Categorical features:", len(categorical_features_day6))

Numerical features: 407
Categorical features: 31


In [13]:
print(
    "Target in training features:",
    "isFraud" in X_train_day6.columns
)

print(
    "Target in validation features:",
    "isFraud" in X_validation_day6.columns
)

Target in training features: False
Target in validation features: False


In [14]:
for feature in new_features:
    print(
        feature,
        "train missing:",
        X_train_day6[feature].isna().sum(),
        "validation missing:",
        X_validation_day6[feature].isna().sum()
    )

V258_high_risk train missing: 0 validation missing: 0
V258_medium_risk train missing: 0 validation missing: 0
V258_missing train missing: 0 validation missing: 0
V294_high train missing: 0 validation missing: 0
V294_missing train missing: 0 validation missing: 0


In [15]:
print("Day 6 raw training shape:", X_train_day6.shape)
print("Day 6 raw validation shape:", X_validation_day6.shape)

Day 6 raw training shape: (484847, 438)
Day 6 raw validation shape: (105693, 438)


In [16]:
print("Numerical features:", len(numerical_features_day6))
print("Categorical features:", len(categorical_features_day6))

Numerical features: 407
Categorical features: 31


In [17]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

# Numerical preprocessing
numerical_imputer_day6 = SimpleImputer(
    strategy="median"
)

X_train_num_day6 = numerical_imputer_day6.fit_transform(
    X_train_day6[numerical_features_day6]
)

X_validation_num_day6 = numerical_imputer_day6.transform(
    X_validation_day6[numerical_features_day6]
)

print(
    "Numerical training:",
    X_train_num_day6.shape
)

print(
    "Numerical validation:",
    X_validation_num_day6.shape
)

Numerical training: (484847, 407)
Numerical validation: (105693, 407)


In [18]:
# Fill categorical missing values
X_train_cat_day6 = X_train_day6[
    categorical_features_day6
].copy()

X_validation_cat_day6 = X_validation_day6[
    categorical_features_day6
].copy()

for feature in categorical_features_day6:

    X_train_cat_day6[feature] = (
        X_train_cat_day6[feature]
        .fillna("__MISSING__")
    )

    X_validation_cat_day6[feature] = (
        X_validation_cat_day6[feature]
        .fillna("__MISSING__")
    )

In [19]:
RARE_THRESHOLD = 50

for feature in categorical_features_day6:

    counts = X_train_cat_day6[
        feature
    ].value_counts()

    frequent_categories = counts[
        counts >= RARE_THRESHOLD
    ].index

    X_train_cat_day6[feature] = (
        X_train_cat_day6[feature].where(
            X_train_cat_day6[feature].isin(
                frequent_categories
            ),
            "__RARE__"
        )
    )

    X_validation_cat_day6[feature] = (
        X_validation_cat_day6[feature].where(
            X_validation_cat_day6[feature].isin(
                frequent_categories
            ),
            "__RARE__"
        )
    )

In [20]:
categorical_encoder_day6 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

X_train_cat_encoded_day6 = (
    categorical_encoder_day6.fit_transform(
        X_train_cat_day6
    )
)

X_validation_cat_encoded_day6 = (
    categorical_encoder_day6.transform(
        X_validation_cat_day6
    )
)

print(
    "Categorical training:",
    X_train_cat_encoded_day6.shape
)

print(
    "Categorical validation:",
    X_validation_cat_encoded_day6.shape
)

Categorical training: (484847, 477)
Categorical validation: (105693, 477)


In [21]:
X_train_final_day6 = hstack([
    csr_matrix(X_train_num_day6),
    X_train_cat_encoded_day6
]).tocsr()

X_validation_final_day6 = hstack([
    csr_matrix(X_validation_num_day6),
    X_validation_cat_encoded_day6
]).tocsr()

print(
    "Day 6 training matrix:",
    X_train_final_day6.shape
)

print(
    "Day 6 validation matrix:",
    X_validation_final_day6.shape
)

Day 6 training matrix: (484847, 884)
Day 6 validation matrix: (105693, 884)


In [22]:
print("Numerical features:", len(numerical_features_day6))
print("Categorical features:", len(categorical_features_day6))

print(
    "Encoded categorical features:",
    X_train_cat_encoded_day6.shape[1]
)

print(
    "Total numerical + categorical:",
    X_train_num_day6.shape[1]
    + X_train_cat_encoded_day6.shape[1]
)

Numerical features: 407
Categorical features: 31
Encoded categorical features: 477
Total numerical + categorical: 884


In [23]:
print(
    "Day 4 expected:",
    891
)

print(
    "Day 6 actual:",
    X_train_final_day6.shape[1]
)

print(
    "Difference:",
    X_train_final_day6.shape[1] - 891
)

Day 4 expected: 891
Day 6 actual: 884
Difference: -7


In [24]:
feature_names_day4 = pd.read_csv(
    PROCESSED_DIR / "feature_names.csv"
)["feature"].values

print("Day 4 feature names:", len(feature_names_day4))

# First 414 features were the numerical features
day4_numerical_features = feature_names_day4[:414]

missing_numerical_features = [
    feature
    for feature in day4_numerical_features
    if feature not in numerical_features_day6
]

print(
    "Missing numerical features:",
    len(missing_numerical_features)
)

print(missing_numerical_features)

Day 4 feature names: 891
Missing numerical features: 12
['transaction_hour', 'transaction_week', 'hour_sin', 'hour_cos', 'identity_present', 'missing_feature_count', 'addr1_missing', 'addr2_missing', 'M1_missing', 'M2_missing', 'M3_missing', 'M6_missing']


In [25]:
SECONDS_PER_HOUR = 60 * 60
SECONDS_PER_DAY = 24 * SECONDS_PER_HOUR

def recreate_day3_features(df):

    df = df.copy()

    # Time features
    df["transaction_hour"] = (
        (df["TransactionDT"] // SECONDS_PER_HOUR) % 24
    )

    df["transaction_week"] = (
        df["transaction_day"] // 7
    )

    # Cyclic time encoding
    df["hour_sin"] = np.sin(
        2 * np.pi * df["transaction_hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["transaction_hour"] / 24
    )

    # Identity availability
    df["identity_present"] = (
        df["id_01"].notna().astype(int)
    )

    # Overall missingness
    missingness_features = [
        col for col in df.columns
        if col not in ["TransactionID", "isFraud"]
    ]

    df["missing_feature_count"] = (
        df[missingness_features].isna().sum(axis=1)
    )

    # Selected missingness indicators
    selected_missing_features = [
        "addr1",
        "addr2",
        "M1",
        "M2",
        "M3",
        "M6"
    ]

    for feature in selected_missing_features:

        df[f"{feature}_missing"] = (
            df[feature].isna().astype(int)
        )

    return df

In [26]:
train_engineered = recreate_day3_features(
    train_original
)

validation_engineered = recreate_day3_features(
    validation_original
)

In [27]:
train_engineered = create_v258_features(
    train_engineered
)

validation_engineered = create_v258_features(
    validation_engineered
)

train_engineered = create_v294_features(
    train_engineered
)

validation_engineered = create_v294_features(
    validation_engineered
)

In [28]:
X_train_day6 = train_engineered.drop(
    columns=["isFraud", "TransactionID"]
)

X_validation_day6 = validation_engineered.drop(
    columns=["isFraud", "TransactionID"]
)

print(
    "Day 6 raw training:",
    X_train_day6.shape
)

print(
    "Day 6 raw validation:",
    X_validation_day6.shape
)

Day 6 raw training: (484847, 450)
Day 6 raw validation: (105693, 450)


In [29]:
numerical_features_day6 = X_train_day6.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features_day6 = X_train_day6.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(
    "Numerical features:",
    len(numerical_features_day6)
)

print(
    "Categorical features:",
    len(categorical_features_day6)
)

Numerical features: 419
Categorical features: 31


In [30]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

numerical_imputer_day6 = SimpleImputer(
    strategy="median"
)

X_train_num_day6 = numerical_imputer_day6.fit_transform(
    X_train_day6[numerical_features_day6]
)

X_validation_num_day6 = numerical_imputer_day6.transform(
    X_validation_day6[numerical_features_day6]
)

print("Numerical training:", X_train_num_day6.shape)
print("Numerical validation:", X_validation_num_day6.shape)

Numerical training: (484847, 419)
Numerical validation: (105693, 419)


In [31]:
X_train_cat_day6 = X_train_day6[
    categorical_features_day6
].copy()

X_validation_cat_day6 = X_validation_day6[
    categorical_features_day6
].copy()

for feature in categorical_features_day6:

    X_train_cat_day6[feature] = (
        X_train_cat_day6[feature]
        .fillna("__MISSING__")
    )

    X_validation_cat_day6[feature] = (
        X_validation_cat_day6[feature]
        .fillna("__MISSING__")
    )

In [32]:
RARE_THRESHOLD = 50

for feature in categorical_features_day6:

    counts = X_train_cat_day6[
        feature
    ].value_counts()

    frequent_categories = counts[
        counts >= RARE_THRESHOLD
    ].index

    X_train_cat_day6[feature] = (
        X_train_cat_day6[feature].where(
            X_train_cat_day6[feature].isin(
                frequent_categories
            ),
            "__RARE__"
        )
    )

    X_validation_cat_day6[feature] = (
        X_validation_cat_day6[feature].where(
            X_validation_cat_day6[feature].isin(
                frequent_categories
            ),
            "__RARE__"
        )
    )

In [33]:
categorical_encoder_day6 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

X_train_cat_encoded_day6 = (
    categorical_encoder_day6.fit_transform(
        X_train_cat_day6
    )
)

X_validation_cat_encoded_day6 = (
    categorical_encoder_day6.transform(
        X_validation_cat_day6
    )
)

print(
    "Categorical training:",
    X_train_cat_encoded_day6.shape
)

print(
    "Categorical validation:",
    X_validation_cat_encoded_day6.shape
)

Categorical training: (484847, 477)
Categorical validation: (105693, 477)


In [34]:
X_train_final_day6 = hstack([
    csr_matrix(X_train_num_day6),
    X_train_cat_encoded_day6
]).tocsr()

X_validation_final_day6 = hstack([
    csr_matrix(X_validation_num_day6),
    X_validation_cat_encoded_day6
]).tocsr()

print(
    "Day 6 training matrix:",
    X_train_final_day6.shape
)

print(
    "Day 6 validation matrix:",
    X_validation_final_day6.shape
)

Day 6 training matrix: (484847, 896)
Day 6 validation matrix: (105693, 896)


In [35]:
from xgboost import XGBClassifier

xgb_day6 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_day6.fit(
    X_train_final_day6,
    y_train,
    eval_set=[(X_validation_final_day6, y_validation)],
    verbose=50
)

[0]	validation_0-aucpr:0.34109
[50]	validation_0-aucpr:0.46709
[100]	validation_0-aucpr:0.49611
[150]	validation_0-aucpr:0.51025
[200]	validation_0-aucpr:0.51894
[250]	validation_0-aucpr:0.52774
[299]	validation_0-aucpr:0.53308


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, ...)

In [36]:
from sklearn.metrics import roc_auc_score, average_precision_score

proba_day6 = xgb_day6.predict_proba(X_validation_final_day6)[:, 1]

roc_day6 = roc_auc_score(y_validation, proba_day6)
pr_day6 = average_precision_score(y_validation, proba_day6)

print(f"ROC-AUC: {roc_day6:.4f}")
print(f"PR-AUC : {pr_day6:.4f}")

ROC-AUC: 0.9038
PR-AUC : 0.5331


In [37]:
from sklearn.metrics import precision_score, recall_score, f1_score

day6_threshold = 0.30

pred_day6 = (
    proba_day6 >= day6_threshold
).astype(int)

print("Threshold:", day6_threshold)

print(
    "Precision:",
    precision_score(
        y_validation,
        pred_day6,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_validation,
        pred_day6,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_validation,
        pred_day6,
        zero_division=0
    )
)

print(
    "Total flagged:",
    pred_day6.sum()
)

Threshold: 0.3
Precision: 0.6261980830670927
Recall: 0.4342287454998615
F1: 0.5128372853638593
Total flagged: 2504


In [38]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    pred_day6
).ravel()

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

True Negatives : 101146
False Positives: 936
False Negatives: 2043
True Positives : 1568
